# 评估指标
- 混淆矩阵
- AUC-ROC
- Recall
- Prcision
- F1-score

# 二分类模型评估：混淆矩阵、ROC与AUC详解

---

## 01 | 二分类混淆矩阵

在二分类问题中，样本根据其真实类别与模型预测类别的组合可分为四种情形：

![alt text](resource/confusion_matrix.png)

由这四个值构成的矩阵称为 **混淆矩阵（Confusion Matrix）**。

### 引申指标

基于混淆矩阵可定义以下两个重要概念：

- **真正例率（True Positive Rate, TPR）** 
  $$
  \text{TPR} = \frac{TP}{TP + FN}
  $$  
  表示所有真实正例中被正确预测的比例，也称 **召回率（Recall）** 或 **灵敏度（Sensitivity）**。  
  ✅ **越大越好**，反映模型识别正样本的能力。

- **假正例率（False Positive Rate, FPR）**  
  $$
  \text{FPR} = \frac{FP}{FP + TN}
  $$  
  表示所有真实负例中被错误预测为正例的比例。  
  ❌ **越小越好**，反映模型对负样本的误判程度。

---

## 02 | ROC曲线（受试者工作特征曲线）

### (1) 预测与分类阈值

- 模型为每个样本输出一个预测得分（如概率值），通过与一个**分类阈值**比较来决定类别：
  - 若得分 > 阈值 → 预测为正类
  - 否则 → 预测为负类

- 因此，在固定阈值下，预测得分的排序直接影响模型泛化能力。

- “最理想”情况：所有正样本排在负样本之前  
  “最坏”情况：所有负样本排在正样本之前

- 分类过程相当于在排序序列中选择一个“截断点”（即分类阈值），前半部分为正例，后半部分为反例。

### (2) 分类阈值的选择策略

- 更重视 **查准率（Precision）** → 选择靠前的截断点（高阈值）
- 更重视 **查全率（Recall）** → 选择靠后的截断点（低阈值）

> 因此，模型的**排序能力**直接决定了其在不同任务下的泛化性能。

### (3) ROC曲线定义

- 全称：**Receiver Operating Characteristic Curve（受试者工作特征曲线）**
- 起源：二战中雷达信号分析技术，用于判断敌机是否存在

#### 坐标轴定义

- 横轴：**假正例率（FPR）**
- 纵轴：**真正例率（TPR）**

> 一个性能越好的模型，其ROC曲线越靠近 **左上角**（高TPR，低FPR）。

### (4) ROC曲线绘制方法

给定 $ M $ 个正例和 $ N $ 个反例：

1. 根据模型预测得分对所有样本进行排序（从小到大）；
2. 将分类阈值设为最大（高于所有得分），此时所有样本预测为负类：
   - 此时 $ \text{TPR} = 0 $，$ \text{FPR} = 0 $，对应坐标原点 $ (0, 0) $
3. 依次将阈值设为每个样本的预测得分，即将该样本及其之前的所有样本划分为正类；
4. 每次调整阈值后，计算当前的 $ \text{TPR} $ 和 $ \text{FPR} $，得到一个点；
   - 若本次新增 $ m $ 个真正例，$ n $ 个假正例，则：
     $$
     \Delta \text{TPR} = \frac{m}{M},\quad \Delta \text{FPR} = \frac{n}{N}
     $$
   - 若前一点为 $ (x, y) $，则新点为：
     $$
     \left( x + \frac{n}{N},\ y + \frac{m}{M} \right)
     $$
5. 将所有点连接起来，形成ROC曲线。

![alt text](resource/ROC.png)
### (5) 极端情况分析

| 情况 | ROC曲线形态 | AUC值 |
|------|-------------|-------|
| **理想模型**：所有正例排在负例之前 | 曲线从(0,0)垂直上升至(0,1)，再水平至(1,1) | AUC = 1 |
| **随机猜测**：正负样本随机交错 | 曲线沿对角线分布（$ y = x $） | AUC = 0.5 |

> ✅ 因此，AUC一般满足：$ 0.5 < \text{AUC} < 1 $

### (6) 模型性能比较

- 若模型A的ROC曲线完全 **“包住”** 模型B → A性能优于B
- 若两条ROC曲线相交 → 无法直接判断优劣，需比较 **AUC（ROC曲线下面积）**

---

## 03 | 基于排序的AUC计算（推荐）

AUC（Area Under the ROC Curve）是ROC曲线下面积，反映模型整体排序能力。

为降低时间复杂度，可通过排序优化计算。

### 计算步骤

1. 将所有样本按预测得分 **从小到大排序**
2. 给每个样本赋予 **rank值**（从1开始），最高得分为 $ M+N $
   - 若多个样本得分相同，则取它们的平均rank
3. 对每个正样本 $ p $，统计其得分 **低于多少个负样本**（即排名在其前面的负样本数）
4. 累加所有正样本对应的负样本数，除以 $ M \times N $

### 公式推导

令 $ \text{rank}_p $ 为第 $ p $ 个正样本的排名（从1开始），$ M $ 为正样本数，$ N $ 为负样本数

则该正样本前面的负样本数量为：  
$$
\text{rank}_p - 1 - (\text{在其前面的正样本数量})
$$

但更简便的方法是：

$$
\text{AUC} = \frac{ \sum_{p \in P} \text{rank}_p - \frac{M(M+1)}{2} }{M \times N}
$$

> ✅ 推导说明：
> - 所有正样本的总排名和为 $ \sum \text{rank}_p $
> - 减去正样本之间的内部排序影响 $ \frac{M(M+1)}{2} $
> - 剩余即为正样本相对于负样本的优势程度

### 时间复杂度

- 排序：$ O((M+N)\log(M+N)) $
- 累加计算：$ O(M+N) $
- 总体：$ O((M+N)\log(M+N)) $

---

## 04 | AUC的Python实现（简要提示）

虽然本文未提供完整代码，但可基于上述方法实现：



In [10]:
import numpy as np
from sklearn.metrics import roc_auc_score

# 方法一：调用sklearn（基于排序法）
y_true = [0, 1, 0, 0, 1]  # 真实标签
y_scores = [0.2, 0.6, 0.8, 0.4, 0.7]  # 预测得分
auc = roc_auc_score(y_true, y_scores)

# 方法二：手动实现（排序法）
def auc_from_ranks(y_true, y_scores):
    
    y_true = np.array(y_true).astype(int)
    y_scores = np.array(y_scores)
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)
    # 按得分升序排序
    sorted_indices = np.argsort(y_scores)
    sorted_labels = np.array(y_true)[sorted_indices]
    # 计算每个正样本的rank（从1开始）
    ranks = np.arange(1, len(y_true) + 1)[sorted_labels == 1]
    # AUC计算
    auc = (np.sum(ranks) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return auc

print(auc)
print(auc_from_ranks(y_true, y_scores))

0.6666666666666667
0.6666666666666666
